In [ ]:
import os

print(os.getenv("GROQ_API_KEY") is not None)

In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

response = model.invoke("Hello, how are you?")

print(response.content)

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [ ]:
class ParentState(TypedDict):

    question: str
    answer_eng: str
    answer_hin: str

In [ ]:
parent_model = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
subgraph_model = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

In [ ]:
def translate_text(state: ParentState):

    prompt = f"""
Translate the following text to Hindi.
Keep it natural and clear. Do not add extra content.

Text:
{state["answer_eng"]}
""".strip()
    
    translated_text = subgraph_model.invoke(prompt).content

    return {'answer_hin': translated_text}

In [ ]:
subgraph_builder = StateGraph(ParentState)

subgraph_builder.add_node('translate_text', translate_text)

subgraph_builder.add_edge(START, 'translate_text')
subgraph_builder.add_edge('translate_text', END)

subgraph = subgraph_builder.compile()

In [ ]:
def generate_answer(state: ParentState):

    answer = parent_model.invoke(f"You are a helpful assistant. Answer clearly.\n\nQuestion: {state['question']}").content
    return {'answer_eng': answer}

In [ ]:
parent_builder = StateGraph(ParentState)

parent_builder.add_node("answer", generate_answer)
parent_builder.add_node("translate", subgraph)

parent_builder.add_edge(START, 'answer')
parent_builder.add_edge('answer', 'translate')
parent_builder.add_edge('translate', END)

In [ ]:
graph = parent_builder.compile()

graph

In [ ]:
graph.invoke({'question': 'What is quantum physics'})